# 05 Advanced

This notebook runs the eight scripts of `05_advanced/` in order with `%run`, so it runs exactly the same code as the command line (`python 05_advanced/NN_name.py`). Each script's docstring says what it shows, what it needs and what it checks.

| # | Script | Tier | Time |
|---|---|---|---|
| 25 | collective effects (JFEL linac) | A; elegant with docker | ~150 s |
| 26 | ASTRA injector from the cathode | B | ~35 s |
| 27 | a ring: tunes, closed orbit, DA, frequency map | A | ~20 s |
| 28 | an energy-recovery linac (multipass) | A; elegant with docker | ~60 s |
| 29 | a plasma-wakefield stage (Wake-T) | A | ~10 s |
| 30 | start-to-end FEL, elegant into Genesis 4 | B | ~40 s |
| 31 | secondaries in BDSIM | C | ~60 s |
| 32 | Bmad, and Bmad multipass | C | ~20 s |

A script whose codes are missing prints `SKIP` (or skips its elegant runs) and carries on. Lines starting `XFAIL #N` are known bugs, listed in `ISSUES.md`. GPT, FBPIC and Ptarmigan are not covered: GPT needs its own licence, FBPIC a GPU, and Ptarmigan support isn't on SIMBA's main yet.

## 25 Collective effects: wakes, CSR, LSC

Every line has `csr_enable`, `lsc_enable` and `wakefield_enable`, and SIMBA passes them to each code in its own terms. **They start on (CSR and wakes) when a `.def` doesn't say**, so set all three when it matters. A code that doesn't model an effect tracks without it, with no error.

JFEL's linac, one effect at a time, compared at the undulator entrance. The wakes take out the chirp before the chicane, so $\sigma_z$ falls by ~40%. CSR in the chicane's bends grows $\varepsilon_{n,x}$ 3–5× and leaves $\varepsilon_{n,y}$ alone. The JFEL README's Ocelot CSR row is stale since SIMBA PR #67 (`ISSUES.md` #29), and the script prints it but doesn't check it.

In [ ]:
%run 25_collective_effects.py

## 26 The injector, from the cathode (ASTRA)

JFEL's ASTRA injector: the generator makes the cathode distribution, `setSeedValue` fixes its random numbers, and ASTRA tracks it to S02, ~35 MeV, 250 pC. The result is compared with the beam laura-lattices saves as `Data_Files/jfel_injector_S02`, which the other JFEL examples start from. At that file's 32768 particles it is reproduced exactly. The default here is 4096, for speed, and then the emittance is within 15% (sampling); `python 26_injector_from_cathode.py --particles 32768` takes ~5 min.

In [ ]:
%run 26_injector_from_cathode.py

## 27 A ring: tunes, closed orbit, dynamic aperture, frequency map

A LAURA section with `geometry: closed` is a ring. `tracking: {periodic: True}` and `ring_parameters()` give the tunes, periodic Twiss and closed orbit; `tracking: {turns, dynamic_aperture}` with `run_dynamic_aperture()` and `run_frequency_map()` track many turns. The ring is SIMBA's own `examples/ring_studies` FODO ring (`ring.py`), and the reference is an independent symplectic tracker in `ring.py`.

The tunes agree everywhere, to $10^{-9}$ with the matrices. The rest has known bugs: Xsuite finds no closed orbit without RF (#30), Ocelot reports a zero orbit (#31), and Ocelot and MAD-X give a dynamic aperture ~30% smaller than Xsuite, elegant and the tracker (#32).

In [ ]:
%run 27_ring_tune_da.py

In [ ]:
from IPython.display import Image, display
display(Image(filename="../output/27_ring_tune_da/dynamic_aperture.png"))

## 28 An energy-recovery linac: one cryomodule, two passes

A LAURA multipass layout: the beam crosses the same eight cavities twice. In a line the linac's elements appear once per pass, `NAME.1` and `NAME.2`, and the second pass's phase is set on the `.2` cavities alone. The beam goes 10 → 100 → 10 MeV/c, the RF takes back all the energy it gave, and the dump beam is the injected beam again: the racetrack is mirror-symmetric.

At the dump screen $\varepsilon_{n,x}$ reads 5 µm. That is the dump dipole's dispersion ($D = -0.58$ m) times the spread, not growth. Take out the $x$–$\delta$ correlation and it is 1.0 µm again. A path-length error, as a phase offset $\phi$ on the return pass, raises the dump momentum as $10 + 90\,(1 - \cos\phi)$ MeV/c.

In [ ]:
%run 28_erl_multipass.py

## 29 A plasma-wakefield stage in Wake-T

A LAURA `Plasma` element, without a laser, is a beam-driven stage. A 1 GeV driver gives up about 15% of what it loses to a 100 MeV witness 80 µm behind it, which reaches ~325 MeV in 1 cm. The rest stays in the wake. `f.change_Lattice_Code("PWFA", "fbpic")` would run the same element in FBPIC (GPU). With 2 mm matching ramps either side, $\beta$ at the exit grows 5×, and the witness's emittance grows less in the drift that follows.

In [ ]:
%run 29_plasma_wake_t.py

## 30 A start-to-end FEL: elegant into Genesis 4

JFEL's linac in elegant (wakes and CSR on), then Genesis 4 steady-state with a 1 kW seed. The Genesis settings are in the line's `fel:` block. The wavelength is the resonance $\lambda = \lambda_u (1 + K^2) / (2\gamma^2)$ = 2.38 µm. The gain length comes out 1.5× Ming Xie's estimate, computed in the script from the beam at the undulator: Ming Xie treats the beam as round, and after CSR $\varepsilon_{n,x}$ is ~5× $\varepsilon_{n,y}$. The time-dependent SASE line `FEL_TD` takes ~4 min and isn't run.

In [ ]:
%run 30_fel_genesis.py

## 31 Secondaries in BDSIM

LAURA `Target` and `Crystal` elements, tracked in BDSIM (Geant4). The line's `tracking:` block names the physics list and the secondaries to keep, and they come out as their own species in the same beam file, `beam.others["photon"]` and so on. A tungsten target is checked against the mean radiation loss $E_0 e^{-t/X_0}$. A converter's positrons go on to Xsuite on their own (`input: {species: positron}`). A bent crystal channels ~9% of the electrons through its 0.9 mrad bend. Geant4's seed changes every run, so the yields vary by a few percent.

In [ ]:
%run 31_secondaries_bdsim.py

## 32 Bmad, and Bmad multipass

28's ERL in Bmad, twice: flattened, as the other codes track it, and with `multipass: native`, where LAURA writes Bmad's own multipass. Each cavity is one lord with a slave per pass, and the return pass's phase is `phi0_multipass` on the slave. The two agree to $10^{-5}$. Needs `ACC_ROOT_DIR` set before Jupyter starts.

In [ ]:
%run 32_bmad_multipass.py